# Astronaut health VLM encoding — the GPU half

Fills the Table 1 / Table 3 rows that do not fit a laptop: **BLIP-2**,
**LLaVA-1.6**, **Qwen2.5-VL** and **Med-Flamingo**.

This notebook does **not** compute any metric. It writes one `.npz` per
(model, dataset) holding per-sample vectors:

| key | meaning |
|---|---|
| `zeroshot_score` | P(exposed) from the yes/no next-token logits |
| `image_embed` | mean-pooled hidden state — what the linear probe reads |
| `labels`, `subjects`, `ids` | copied from the manifest |

Download those files, drop them into `data/vlm_astro/<dataset>/encodings/`,
and every table recomputes on CPU in seconds:

```bash
python vlm_bench/astro_eval.py table1
python vlm_bench/astro_eval.py table3
```

**Turn the accelerator on** (Settings → Accelerator → GPU T4 x2) and leave
internet enabled for the model downloads.

## Getting the data here

`data/` is gitignored, so the rendered panels have to be uploaded. They are
small (~6 MB):

```bash
python vlm_bench/astro_modalities.py     # writes data/vlm_astro/
# then upload the whole data/vlm_astro/ folder as a Kaggle Dataset
```

Attach it via **Add Input** and set `ASTRO_INPUT` below to its folder name.
Cell 1 searches `/kaggle/input` for `manifest.json` if you leave it blank.

In [ ]:
# 0. Config
ASTRO_INPUT = ""       # Kaggle dataset folder holding vlm_astro/, blank = autodetect
MODELS = ["blip2", "llava16", "qwen25vl"]   # add "medflamingo" once its weights resolve
DATASETS = ["inspiration4", "genelab_astronaut", "osdr_bedrest"]
MAX_NEW_TOKENS = 1
OUT_DIR = "/kaggle/working/encodings"

import os, json, pathlib, sys
os.makedirs(OUT_DIR, exist_ok=True)


def find_manifests():
    """Locate one manifest.json per dataset, wherever the input got mounted."""
    roots = []
    if ASTRO_INPUT:
        roots.append(pathlib.Path(f"/kaggle/input/{ASTRO_INPUT}"))
    roots += [pathlib.Path("/kaggle/input"), pathlib.Path("/kaggle/working")]
    found = {}
    for root in roots:
        if not root.is_dir():
            continue
        for hit in sorted(root.rglob("manifest.json")):
            try:
                key = json.loads(hit.read_text())["dataset"]
            except Exception:
                continue
            found.setdefault(key, hit)
    return found


MANIFESTS = find_manifests()
print("found manifests:", {k: str(v) for k, v in MANIFESTS.items()})
missing = [d for d in DATASETS if d not in MANIFESTS]
if missing:
    print(f"\nMISSING {missing} — attach the vlm_astro dataset via Add Input.")

In [ ]:
# 1. Dependencies
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.49.0", "accelerate>=0.33.0", "bitsandbytes>=0.43.0",
    "pillow", "einops", "qwen-vl-utils", "sentencepiece", "protobuf",
])

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU. Settings -> Accelerator -> GPU T4 x2.")

In [ ]:
# 2. Shared helpers — identical conventions to vlm_bench/astro_eval.py
import numpy as np
from PIL import Image


def load_samples(dataset):
    """Manifest entries plus resolved image paths.

    image_path is stored repo-relative; on Kaggle the images sit beside the
    manifest, so resolve against the manifest's own directory.
    """
    path = MANIFESTS[dataset]
    manifest = json.loads(path.read_text())
    base = path.parent
    for sample in manifest["samples"]:
        name = pathlib.Path(sample["image_path"]).name
        candidate = base / "images" / name
        sample["resolved"] = candidate if candidate.exists() else base / name
    return manifest


def yes_no_probability(logits, tokenizer):
    """P(yes) / (P(yes) + P(no)) from the next-token logits.

    Same trick as vlm_bench/eval_models.py: scoring the two answer tokens
    directly keeps a real probability for AUROC, instead of parsing free-form
    generation and losing the ranking.
    """
    yes_id = tokenizer.encode("Yes", add_special_tokens=False)[0]
    no_id = tokenizer.encode("No", add_special_tokens=False)[0]
    pair = torch.stack([logits[no_id], logits[yes_id]])
    return float(pair.softmax(dim=-1)[1])


def question_for(manifest):
    """Turn the manifest's positive prompt into a yes/no question."""
    positive = manifest["prompts"][1]
    return (
        "This heatmap encodes one astronaut biomedical sample, one cell per "
        f"measurement. Is this {positive}? Answer Yes or No."
    )


def save(model_name, dataset, manifest, scores, embeds):
    samples = manifest["samples"]
    out = pathlib.Path(OUT_DIR) / dataset
    out.mkdir(parents=True, exist_ok=True)
    path = out / f"{model_name}.npz"
    embeds = np.asarray(embeds, dtype=np.float32)
    np.savez_compressed(
        path,
        ids=np.array([s["id"] for s in samples]),
        labels=np.array([s["label"] for s in samples], dtype=int),
        subjects=np.array([s["subject_id"] for s in samples]),
        image_embed=embeds,
        # No separate text tower in a generative VLM: the embedding already
        # conditions on the prompt, so text_embed mirrors it and the RQ4
        # ablation stays a BiomedCLIP-only result.
        text_embed=embeds,
        physio=np.array([s["physio"] for s in samples], dtype=np.float32),
        zeroshot_score=np.asarray(scores, dtype=np.float32),
    )
    from sklearn.metrics import roc_auc_score
    labels = [s["label"] for s in samples]
    auroc = roc_auc_score(labels, scores) if len(set(labels)) > 1 else float("nan")
    print(f"  saved {path}  (zero-shot AUROC {auroc:.3f})")

## BLIP-2 — Table 1 row 2, Table 3 row 2

In [ ]:
def encode_blip2(dataset):
    from transformers import Blip2ForConditionalGeneration, Blip2Processor

    name = "Salesforce/blip2-opt-2.7b"
    processor = Blip2Processor.from_pretrained(name)
    model = Blip2ForConditionalGeneration.from_pretrained(
        name, torch_dtype=torch.float16, device_map="auto",
    ).eval()

    manifest = load_samples(dataset)
    question = question_for(manifest)
    scores, embeds = [], []
    for sample in manifest["samples"]:
        image = Image.open(sample["resolved"]).convert("RGB")
        inputs = processor(
            images=image, text=f"Question: {question} Answer:", return_tensors="pt",
        ).to(DEVICE, torch.float16)
        with torch.inference_mode():
            out = model(**inputs, output_hidden_states=True)
            scores.append(yes_no_probability(out.logits[0, -1].float(), processor.tokenizer))
            embeds.append(out.hidden_states[-1][0].float().mean(0).cpu().numpy())
    save("blip2", dataset, manifest, scores, embeds)
    del model
    torch.cuda.empty_cache()


if "blip2" in MODELS:
    for dataset in DATASETS:
        if dataset in MANIFESTS:
            print(f"blip2 x {dataset}")
            encode_blip2(dataset)

## LLaVA-1.6 — Table 1 row 3

4-bit so the 7B fits a single T4 alongside everything else.

In [ ]:
def encode_llava(dataset):
    from transformers import (
        BitsAndBytesConfig, LlavaNextForConditionalGeneration, LlavaNextProcessor,
    )

    name = "llava-hf/llava-v1.6-mistral-7b-hf"
    processor = LlavaNextProcessor.from_pretrained(name)
    model = LlavaNextForConditionalGeneration.from_pretrained(
        name,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
        ),
        device_map="auto",
    ).eval()

    manifest = load_samples(dataset)
    question = question_for(manifest)
    scores, embeds = [], []
    for sample in manifest["samples"]:
        image = Image.open(sample["resolved"]).convert("RGB")
        prompt = f"[INST] <image>\n{question} [/INST]"
        inputs = processor(images=image, text=prompt, return_tensors="pt").to(DEVICE)
        with torch.inference_mode():
            out = model(**inputs, output_hidden_states=True)
            scores.append(yes_no_probability(out.logits[0, -1].float(), processor.tokenizer))
            embeds.append(out.hidden_states[-1][0].float().mean(0).cpu().numpy())
    save("llava16", dataset, manifest, scores, embeds)
    del model
    torch.cuda.empty_cache()


if "llava16" in MODELS:
    for dataset in DATASETS:
        if dataset in MANIFESTS:
            print(f"llava16 x {dataset}")
            encode_llava(dataset)

## Qwen2.5-VL — Table 1 row 4

The 3B fits a T4 far more reliably than the 7B.

In [ ]:
def encode_qwen(dataset):
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

    name = "Qwen/Qwen2.5-VL-3B-Instruct"
    processor = AutoProcessor.from_pretrained(name)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        name, torch_dtype=torch.float16, device_map="auto",
    ).eval()

    manifest = load_samples(dataset)
    question = question_for(manifest)
    scores, embeds = [], []
    for sample in manifest["samples"]:
        image = Image.open(sample["resolved"]).convert("RGB")
        messages = [{"role": "user", "content": [
            {"type": "image"}, {"type": "text", "text": question},
        ]}]
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )
        inputs = processor(text=[text], images=[image], return_tensors="pt").to(DEVICE)
        with torch.inference_mode():
            out = model(**inputs, output_hidden_states=True)
            scores.append(yes_no_probability(out.logits[0, -1].float(), processor.tokenizer))
            embeds.append(out.hidden_states[-1][0].float().mean(0).cpu().numpy())
    save("qwen25vl", dataset, manifest, scores, embeds)
    del model
    torch.cuda.empty_cache()


if "qwen25vl" in MODELS:
    for dataset in DATASETS:
        if dataset in MANIFESTS:
            print(f"qwen25vl x {dataset}")
            encode_qwen(dataset)

## Med-Flamingo — Table 3 row 4

Off by default. Its declared base model `decapoda-research/llama-7b-hf` was
taken down; `vlm_bench/RESULTS.md` documents `huggyllama/llama-7b` as the
standard replacement, and the `open_flamingo` wiring it needs is heavier than
the three above. Add `"medflamingo"` to `MODELS` only after confirming the
weights resolve — otherwise the Table 3 row stays `missing_encoding`, which is
the honest state.

In [ ]:
# 3. Package the encodings for download
import shutil

produced = sorted(pathlib.Path(OUT_DIR).rglob("*.npz"))
for path in produced:
    print(f"{path.relative_to(OUT_DIR)}  {path.stat().st_size / 1024:.0f} KB")

if produced:
    shutil.make_archive("/kaggle/working/astro_encodings", "zip", OUT_DIR)
    print("\nDownload astro_encodings.zip, then locally:")
    print("  unzip -o astro_encodings.zip -d /tmp/enc")
    print("  for d in /tmp/enc/*; do")
    print("    mkdir -p data/vlm_astro/$(basename $d)/encodings")
    print("    cp $d/*.npz data/vlm_astro/$(basename $d)/encodings/")
    print("  done")
    print("  python vlm_bench/astro_eval.py table1")
    print("  python vlm_bench/astro_eval.py table3")
else:
    print("No encodings produced — check the manifest paths in cell 0.")